In [ ]:
# Installation des dépendances (à exécuter une seule fois)
!pip install -U "bitsandbytes>=0.46.1" transformers pandas

import json
import re
import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Chargement du modèle Qwen2.5-Math-7B-Instruct (Optimisé 4-bit)
model_name = "Qwen/Qwen2.5-7B-Instruct"
quant_config = BitsAndBytesConfig(load_in_4bit=True)

print("Chargement du tokenizer et du modèle...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)
print("Modèle chargé avec succès !")

In [ ]:
# ============================================================
# PROMPTS POUR LE PIPELINE CoVe
# ============================================================

# 1. BASELINE PROMPT : Le modèle donne une première analyse (avec raisonnement libre)
BASELINE_PROMPT = """You are a mathematics expert evaluating whether a proposed list of hypotheses satisfies the requirements of a theorem.
Theorem: {theorem_name}
Proposed hypotheses:
{hypotheses_list}

Analyze the proposed hypotheses against the standard required hypotheses for this theorem. 
Identify if it is an exact match, a valid implication (stronger hypothesis), a missing hypothesis, multiple missing, or an invented hypothesis.
Provide your reasoning."""

# 2. PLANNING PROMPT : Le modèle génère des questions de vérification
PLANNING_PROMPT = """Based on the following theorem, proposed hypotheses, and baseline evaluation, generate 3 specific, independent verification questions to fact-check the evaluation. 
Focus on:
1. What are the exact standard hypotheses required for this theorem?
2. Does a specific proposed hypothesis mathematically imply the required one?
3. Is any required hypothesis completely missing?

Theorem: {theorem_name}
Proposed hypotheses:
{hypotheses_list}
Baseline Evaluation:
{baseline_text}

Return ONLY a numbered list of 3 questions. Do not answer them."""

# 3. EXECUTION PROMPT : Le modèle répond à une question isolée (Factored)
EXECUTION_PROMPT = """You are a mathematical verifier. Answer the following question accurately and concisely based purely on mathematical facts:
Question: {question}"""

# 4. FINAL PROMPT : Le modèle génère le JSON final en prenant en compte les vérifications
FINAL_SYSTEM_PROMPT = """You are a mathematics expert evaluating whether a proposed list of hypotheses correctly satisfies the hypotheses required by a theorem.
You must classify the VALIDATION TYPE into one of: ["correct", "valid_implication", "missing_hypothesis", "multiple_missing", "invented_hypothesis"].

You are provided with a baseline evaluation and a set of verified facts (Q&A). 
Use these verified facts to CORRECT any hallucinations or errors in the baseline evaluation.

OUTPUT FORMAT:
Return ONLY a JSON object:
{
  "label": "correct" or "incorrect",
  "validation_type": "correct" or "valid_implication" or "missing_hypothesis" or "multiple_missing" or "invented_hypothesis",
  "reason": "concise mathematical explanation based on the verified facts"
}
CRITICAL: Return ONLY the JSON object. No markdown. Start with "{" and end with "}"."""

FINAL_USER_PROMPT = """Theorem: {theorem_name}
Proposed hypotheses:
{hypotheses_list}

Baseline Evaluation (may contain errors):
{baseline_text}

Verified Facts (use these to correct the baseline if needed):
{qa_pairs}

Provide the final JSON evaluation."""

# ============================================================
# FONCTION D'APPEL AU LLM
# ============================================================
def call_llm(system_msg, user_msg, max_tokens=500):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# ============================================================
# PIPELINE CoVe COMPLET
# ============================================================
def run_cove_pipeline(theorem_name, hypotheses):
    hyp_text = "\n".join(f"- {h}" for h in hypotheses)
    
    # ÉTAPE 1 : Baseline
    baseline_text = call_llm(
        "You are a helpful math assistant.", 
        BASELINE_PROMPT.format(theorem_name=theorem_name, hypotheses_list=hyp_text)
    )
    
    # ÉTAPE 2 : Planification (Génération des questions)
    planning_text = call_llm(
        "You are an expert at fact-checking mathematical statements.",
        PLANNING_PROMPT.format(theorem_name=theorem_name, hypotheses_list=hyp_text, baseline_text=baseline_text)
    )
    
    # Extraction naïve des questions (basée sur les retours à la ligne)
    questions = [q.strip() for q in planning_text.split('\n') if q.strip() and q[0].isdigit()]
    if not questions:
        questions = [planning_text] # Fallback si le format n'est pas une liste numérotée
        
    # ÉTAPE 3 : Exécution (Factored - Une requête par question)
    qa_pairs = ""
    for q in questions:
        answer = call_llm(
            "You are a strict mathematical verifier.",
            EXECUTION_PROMPT.format(question=q),
            max_tokens=150
        )
        qa_pairs += f"Q: {q}\nA: {answer}\n\n"
        
    # ÉTAPE 4 : Réponse Finale Corrigée
    final_raw = call_llm(
        FINAL_SYSTEM_PROMPT,
        FINAL_USER_PROMPT.format(
            theorem_name=theorem_name, 
            hypotheses_list=hyp_text, 
            baseline_text=baseline_text, 
            qa_pairs=qa_pairs
        ),
        max_tokens=300
    )
    
    return final_raw, baseline_text, qa_pairs

In [ ]:
import re

def extract_from_broken_json(raw_text):
    if not isinstance(raw_text, str):
        return None, None, None, "No response"
        
    # Extraction de force via Regex pour contourner les erreurs de syntaxe JSON
    label_match = re.search(r'["\']label["\']\s*:\s*["\']([a-zA-Z_]+)["\']', raw_text)
    val_match = re.search(r'["\']validation_type["\']\s*:\s*["\']([a-zA-Z_]+)["\']', raw_text)
    
    label = label_match.group(1).lower() if label_match else None
    val_type = val_match.group(1).lower() if val_match else None
    
    if val_type == "correct":
        val_type = "exact_match"
        
    if label and val_type:
        return label, val_type, "Extrait de force (erreur JSON initiale contournée)", None
        
    return None, None, None, "Échec de l'extraction Regex"

# Affichage de l'erreur et réparation du DataFrame
erreurs = df_cove[df_cove['parse_error'].notna()]

for idx, row in erreurs.iterrows():
    raw = row['raw_response']
    print(f"--- Ce que le LLM a généré pour l'index {idx} ({row['name']}) ---")
    print(raw)
    print("-" * 70)
    
    # Réparation
    label, val_type, reason, err = extract_from_broken_json(raw)
    df_cove.loc[idx, 'prediction_label'] = label
    df_cove.loc[idx, 'prediction_validation'] = val_type
    df_cove.loc[idx, 'prediction_reason'] = reason
    df_cove.loc[idx, 'parse_error'] = err

# Sauvegarde du fichier CSV corrigé
df_cove.to_csv(OUTPUT_CSV_PATH, index=False)

# Affichage du résultat corrigé
display(df_cove[['name', 'ground_truth_validation', 'prediction_validation', 'parse_error']].head())

In [ ]:
import json

import json
import re

def parse_llm_output(raw_text):
    if not raw_text:
        return {"label": None, "validation_type": None, "reason": None, "parse_error": "no_response"}
    
    # Nettoyage des balises markdown
    cleaned = re.sub(r"```(?:json)?", "", raw_text, flags=re.IGNORECASE).strip()
    
    # Utilisation d'un décodeur robuste
    decoder = json.JSONDecoder()
    
    # On cherche chaque accolade ouvrante
    for match in re.finditer(r"\{", cleaned):
        start = match.start()
        try:
            # raw_decode extrait le premier objet JSON valide à partir de 'start'
            parsed, end = decoder.raw_decode(cleaned[start:])
            
            val_type = parsed.get("validation_type")
            if val_type == "correct":
                val_type = "exact_match" # Alignement avec la taxonomie
                
            return {
                "label": parsed.get("label"),
                "validation_type": val_type,
                "reason": parsed.get("reason"),
                "parse_error": None,
            }
        except json.JSONDecodeError:
            continue
            
    return {"label": None, "validation_type": None, "reason": None, "parse_error": "JSON decode error"}
# ============================================================
# EXÉCUTION DU BENCHMARK AVEC CoVe
# ============================================================

INPUT_JSON_PATH = "/kaggle/input/datasets/mathurasanthalingam/benchmark-students-copy/benchmark_data.json"
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_cove_results.csv"

with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Pour tester sans attendre des heures, on peut limiter aux 10 premiers exemples :
dataset = dataset[:10]
results = []
start_time = time.time()

print(f"Lancement de CoVe sur {len(dataset)} exemples...")

for i, sample in enumerate(dataset):
    theorem_name = sample["name"]
    hypotheses = sample["copy"]
    
    try:
        final_raw, baseline, qa_pairs = run_cove_pipeline(theorem_name, hypotheses)
        parsed = parse_llm_output(final_raw)
    except Exception as e:
        final_raw, baseline, qa_pairs = str(e), "", ""
        parsed = {"label": None, "validation_type": None, "reason": None, "parse_error": str(e)}

    results.append({
        "theorem_id": sample["theorem_id"],
        "name": theorem_name,
        "ground_truth_label": "correct" if sample["is_correct"] else "incorrect",
        "ground_truth_validation": sample["validation_type"],
        
        "prediction_label": parsed["label"],
        "prediction_validation": parsed["validation_type"],
        "prediction_reason": parsed["reason"],
        
        "cove_baseline": baseline,
        "cove_qa_pairs": qa_pairs,
        "parse_error": parsed["parse_error"],
        "raw_response": final_raw,
    })
    
    if (i + 1) % 5 == 0:
        print(f"{i+1}/{len(dataset)} exemples traités...")

# Sauvegarde
df_cove = pd.DataFrame(results)
df_cove.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"\nTerminé ! Sauvegardé dans {OUTPUT_CSV_PATH}")
print(f"Temps d'exécution : {(time.time() - start_time) / 60:.2f} minutes")

# Affichage d'un aperçu
display(df_cove[['name', 'ground_truth_validation', 'prediction_validation', 'parse_error']].head())

In [1]:
pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 36.5 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


### charger le model Qwen2.5-Math-7B-Instruct

In [2]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
model_name = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

### Evaluation for the entire dataset


In [3]:
"""
Benchmark de vérification d'hypothèses de théorèmes mathématiques
avec Qwen2.5-7B-Instruct (4-bit) sur Kaggle.

Le fichier JSON doit contenir une liste d'objets du type :

{
    "theorem_id": "T21",
    "name": "Adjacent Sequences Theorem",
    "copy": [...],
    "expected": [...],
    "is_correct": false,
    "reason": "...",
    "validation_type": "missing_hypothesis"
}

La taxonomie des validations est :

- correct
- valid_implication
- missing_hypothesis
- multiple_missing
- invented_hypothesis

"correct" et "valid_implication" correspondent à un verdict global correct.
"""

import json
import re
import time
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# Adapter selon ton dataset Kaggle

INPUT_JSON_PATH = "/kaggle/input/datasets/mathurasanthalingam/benchmark/benchmark_data.json"
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_results.csv"

# ============================================================
# CHARGEMENT DU MODELE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(model_name)

# ============================================================
# PROMPTS
# ============================================================

SYSTEM_PROMPT = """You are a mathematics expert evaluating whether a proposed list of hypotheses correctly satisfies the hypotheses required by a theorem.

Your task is to classify the VALIDATION TYPE of the proposed list.

The benchmark distinguishes the following cases:

- "correct": the proposed hypotheses match the required hypotheses exactly.
- "valid_implication": the proposed hypotheses correctly REPLACE one or more required hypotheses by strictly stronger hypotheses that logically imply them.
- "missing_hypothesis": exactly one required hypothesis is not satisfied.
- "multiple_missing": more than one required hypothesis is not satisfied.
- "invented_hypothesis": at least one proposed hypothesis does not correctly satisfy any required hypothesis.

IMPORTANT: "valid_implication" is a CORRECT case, not an error.

EVALUATION RULES:

1. Identify the standard mathematical hypotheses required by the theorem.

2. Each required hypothesis must be satisfied either:
   - directly by the corresponding proposed hypothesis, or
   - by a proposed hypothesis that is strictly stronger and logically implies it.

3. A stronger hypothesis is accepted ONLY if the implication is mathematically valid on its own, without requiring an additional unstated assumption.

4. A stronger hypothesis may REPLACE a required hypothesis.

   Example:
   Required: "f is continuous on [a,b]"
   Proposed: "f is differentiable on [a,b]"

   Since differentiable => continuous, the proposed hypothesis validly replaces the required one.

5. A proposed hypothesis that is merely RELATED to a required hypothesis is NOT sufficient.

   Example:
   Required: "F is a closed subspace of E"
   Proposed: "F is a subspace of E"

   This is NOT a valid implication because being a subspace does not imply being closed.

   Therefore this must NOT be classified as "valid_implication".

6. A replacement must preserve the logical requirement of the theorem.

   If the proposed hypothesis weakens, changes, restricts, or alters a required condition in a way that no longer implies the required condition, it does not satisfy that requirement.

7. If a proposed hypothesis is not needed to satisfy any required hypothesis, it is an invented hypothesis.

8. IMPORTANT DISTINCTION BETWEEN REPLACEMENT AND OMISSION:

   If a proposed hypothesis replaces a required hypothesis with another hypothesis, there is an EXTRA proposed hypothesis that must be evaluated.

   If that proposed hypothesis does not imply the required one, classify it as "invented_hypothesis".

   Do NOT reinterpret such a case as both "missing_hypothesis" and "invented_hypothesis".

   The benchmark treats a replacement by an incorrect hypothesis as "invented_hypothesis".

9. The distinction between "missing_hypothesis" and "invented_hypothesis" is therefore:

   - "missing_hypothesis": a required hypothesis is simply absent, and no incorrect replacement occupies its place.
   - "multiple_missing": several required hypotheses are simply absent, and no incorrect replacements occupy their places.
   - "invented_hypothesis": a proposed hypothesis replaces, alters, or adds a condition that does not correctly satisfy a required hypothesis.

10. Do not assume hypotheses that are not explicitly stated.

11. Do not add mathematical assumptions merely because they are usually understood informally.

12. Do not judge whether the theorem itself is true or false. Only evaluate whether the proposed hypothesis list correctly satisfies the standard hypotheses of the theorem.

13. Redundant hypotheses are invented hypotheses if they are not required and do not serve as a replacement for a distinct required hypothesis.

14. Exact equality of the proposed list and the required list is "correct".

DECISION PROCEDURE (Apply these steps strictly in order):

Step 1 (Ground Truth): List the exact hypotheses required by the standard mathematical theorem.

Step 2 (Exact Match Check): Are the proposed hypotheses strictly identical to the required hypotheses, with no additions or omissions?
If YES -> validation_type = "correct". (Stop here).

Step 3 (Missing Check): Is there any required hypothesis that is completely absent AND has not been replaced by any other statement?
If exactly ONE is absent -> validation_type = "missing_hypothesis". (Stop here).
If MORE THAN ONE is absent -> validation_type = "multiple_missing". (Stop here).

Step 4 (Implication vs Invention Check): You are left with hypotheses that don't match the required ones exactly. Evaluate each non-matching proposed hypothesis:
- Does it logically and mathematically imply a required hypothesis without needing unstated assumptions? If YES, it's a valid replacement.
- If it does NOT logically imply a required hypothesis, or if it is completely unnecessary/redundant, it is an invention.

Step 5 (Final Classification):
- If ALL non-matching proposed hypotheses are valid replacements (and all requirements are met) -> validation_type = "valid_implication".
- If AT LEAST ONE proposed hypothesis does NOT imply a required hypothesis (it is incorrect, extra, or redundant) -> validation_type = "invented_hypothesis".

CRITICAL PRIORITY: Never classify as "valid_implication" if an essential condition of the theorem is missing or violated.

OUTPUT FORMAT:

Return ONLY this JSON object:

{
  "label": "correct" or "incorrect",
  "validation_type": "correct" or "valid_implication" or "missing_hypothesis" or "multiple_missing" or "invented_hypothesis",
  "reason": "concise mathematical explanation"
}

The label must be:

- "correct" for validation_type = "correct" or "valid_implication"
- "incorrect" for validation_type = "missing_hypothesis", "multiple_missing", or "invented_hypothesis"

CRITICAL:
You MUST return ONLY the JSON object.
No markdown.
No explanation before or after the JSON.
Start with "{" and end with "}".

"""


USER_PROMPT_TEMPLATE = """Theorem: {theorem_name}

Proposed hypotheses:
{hypotheses_list}

Evaluate whether this list correctly covers the hypotheses required by the standard version of the theorem.

Remember:
- A stronger hypothesis may REPLACE a required hypothesis if it logically implies it.
- A valid stronger replacement is classified as "valid_implication" and is CORRECT.
- An incorrect replacement is classified as "invented_hypothesis".
- A replacement must NOT be decomposed into a missing hypothesis plus an invented hypothesis.
- An unnecessary additional hypothesis is "invented_hypothesis".

CRITICAL: Return ONLY the JSON object, starting with "{{" and ending with "}}". No explanation.
"""


def build_user_prompt(theorem_name, hypotheses):
    hyp = "\n".join(f"- {h}" for h in hypotheses)

    return USER_PROMPT_TEMPLATE.format(
        theorem_name=theorem_name,
        hypotheses_list=hyp,
    )


# ============================================================
# APPEL LLM
# ============================================================

def call_llm(theorem_name, hypotheses):

    user_prompt = build_user_prompt(
        theorem_name,
        hypotheses,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return answer.strip()


# ============================================================
# PARSER JSON
# ============================================================

def parse_llm_output(raw_text):

    if raw_text is None:
        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": "no_response",
        }

    cleaned = raw_text.strip()

    # --------------------------------------------------------
    # 1. Nettoyage éventuel des balises Markdown
    # --------------------------------------------------------

    cleaned = re.sub(
        r"```(?:json)?",
        "",
        cleaned,
        flags=re.IGNORECASE,
    ).strip()

    cleaned = cleaned.replace("```", "").strip()

    # --------------------------------------------------------
    # 2. Chercher des objets JSON candidats
    #
    # On évite le greedy match "{.*}" qui pouvait englober
    # une grande quantité de texte non JSON.
    # --------------------------------------------------------

    decoder = json.JSONDecoder()

    candidates = []

    for match in re.finditer(r"\{", cleaned):

        start = match.start()

        try:
            obj, end = decoder.raw_decode(cleaned[start:])
            candidates.append(obj)
        except json.JSONDecodeError:
            continue

    # --------------------------------------------------------
    # 3. Aucun JSON trouvé
    # --------------------------------------------------------

    if not candidates:
        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": "no_valid_json_object_found",
        }

    # --------------------------------------------------------
    # 4. Chercher de préférence un objet ayant la bonne structure
    # --------------------------------------------------------

    parsed = None

    for candidate in candidates:

        if not isinstance(candidate, dict):
            continue

        if (
            "label" in candidate
            and "validation_type" in candidate
            and "reason" in candidate
        ):
            parsed = candidate
            break

    # Si aucun objet ne possède exactement les champs attendus,
    # on prend malgré tout le premier dictionnaire trouvé.
    parsed = None
    for candidate in candidates:
        if not isinstance(candidate, dict):
            continue
        if "label" in candidate and "validation_type" in candidate and "reason" in candidate:
            parsed = candidate
            break
    
    if parsed is None:
        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": "no_object_with_required_keys",
        }

    # --------------------------------------------------------
    # 5. Validation minimale du contenu
    # --------------------------------------------------------

    label = parsed.get("label")
    validation_type = parsed.get("validation_type")
    reason = parsed.get("reason")

    valid_labels = {"correct", "incorrect"}

    # On AJOUTE "correct" à la liste des mots autorisés du LLM
    valid_validation_types = {
        "correct",               # Le mot demandé au LLM dans le prompt
        "exact_match",           # (Au cas où il l'utiliserait quand même)
        "stronger_hypothesis",
        "missing_hypothesis",
        "multiple_missing",
        "invented_hypothesis",
        "valid_implication",
        None,
    }

    if label not in valid_labels:
        return {
            "label": label,
            "validation_type": validation_type,
            "reason": reason,
            "parse_error": f"invalid_label: {label}",
        }

    if validation_type not in valid_validation_types:
        return {
            "label": label,
            "validation_type": validation_type,
            "reason": reason,
            "parse_error": (
                f"invalid_validation_type: {validation_type}"
            ),
        }

    # TRADUCTION POUR LE BENCHMARK :
    # Si le LLM a renvoyé "correct" comme type de validation, on le
    # traduit en "exact_match" pour que ça corresponde parfaitement
    # au vocabulaire de votre colonne ground_truth_validation.
    if validation_type == "correct":
        validation_type = "exact_match"

    return {
        "label": label,
        "validation_type": validation_type,
        "reason": reason,
        "parse_error": None,
    }

# ============================================================
# CHARGEMENT DATASET
# ============================================================

print("Loading dataset...")

with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"{len(dataset)} examples loaded.")


# ============================================================
# BENCHMARK
# ============================================================

results = []

start = time.time()

for i, sample in enumerate(dataset):

    theorem_name = sample["name"]
    hypotheses = sample["copy"]

    try:

        raw = call_llm(
            theorem_name,
            hypotheses,
        )

        parsed = parse_llm_output(raw)

    except Exception as e:

        raw = None

        parsed = {
            "label": None,

            # [MODIFICATION]
            "validation_type": None,

            "reason": None,
            "parse_error": str(e),
        }

    results.append({

        "theorem_id": sample["theorem_id"],

        "name": theorem_name,

        "hypotheses": sample["copy"],

        "expected": sample["expected"],

        "ground_truth_label":
            "correct" if sample["is_correct"] else "incorrect",

        # [MODIFICATION]
        # On récupère désormais le type de validation produit par le
        # générateur du benchmark.
        "ground_truth_validation":
            sample["validation_type"],

        "ground_truth_reason":
            sample["reason"],

        "prediction_label":
            parsed["label"],

        # [MODIFICATION]
        "prediction_validation":
            parsed["validation_type"],

        "prediction_reason":
            parsed["reason"],

        "parse_error":
            parsed["parse_error"],

        "raw_response":
            raw,
    })

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(dataset)} examples processed")


# ============================================================
# SAUVEGARDE
# ============================================================

df = pd.DataFrame(results)

df.to_csv(
    OUTPUT_CSV_PATH,
    index=False,
)

print()
print("CSV saved to:")
print(OUTPUT_CSV_PATH)


# ============================================================
# MÉTRIQUES
# ============================================================

valid = df[df["parse_error"].isna()].copy()

# ------------------------------------------------------------
# 1. Accuracy du verdict global
# ------------------------------------------------------------

label_accuracy = (
    valid["ground_truth_label"]
    == valid["prediction_label"]
).mean()


# ------------------------------------------------------------
# 2. Accuracy du validation_type sur TOUS les exemples
# ------------------------------------------------------------

validation_type_accuracy = (
    valid["ground_truth_validation"]
    == valid["prediction_validation"]
).mean()


# ------------------------------------------------------------
# 3. Accuracy du type d'erreur sur les CAS INCORRECTS
# ------------------------------------------------------------

incorrect = valid[
    valid["ground_truth_label"] == "incorrect"
].copy()

error_type_accuracy = (
    incorrect["ground_truth_validation"]
    == incorrect["prediction_validation"]
).mean()


# ------------------------------------------------------------
# 4. Accuracy de la justification des CAS CORRECTS
# ------------------------------------------------------------

correct = valid[
    valid["ground_truth_label"] == "correct"
].copy()

correct_type_accuracy = (
    correct["ground_truth_validation"]
    == correct["prediction_validation"]
).mean()


print()
print("=" * 60)
print("RESULTS")
print("=" * 60)

print(f"Examples                    : {len(df)}")
print(f"Valid generations           : {len(valid)}")
print(f"Label accuracy              : {label_accuracy:.4f}")
print(f"Validation type accuracy    : {validation_type_accuracy:.4f}")
print(f"Error type accuracy         : {error_type_accuracy:.4f}")
print(f"Correct-case accuracy       : {correct_type_accuracy:.4f}")
print(f"Execution time              : {(time.time()-start)/60:.2f} min")
# [MODIFICATION]
# Métrique supplémentaire : parmi les exemples mathématiquement corrects,
# le LLM distingue-t-il correctement "correct" de "valid_implication" ?
correct_validation_accuracy = (
    valid["ground_truth_validation"].isin(
        ["correct", "valid_implication"]
    )
    ==
    valid["prediction_validation"].isin(
        ["correct", "valid_implication"]
    )
).mean()

display(df.head())

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading dataset...
100 examples loaded.
10/100 examples processed
20/100 examples processed
30/100 examples processed
40/100 examples processed
50/100 examples processed
60/100 examples processed
70/100 examples processed
80/100 examples processed
90/100 examples processed
100/100 examples processed

CSV saved to:
/kaggle/working/benchmark_results.csv

RESULTS
Examples                    : 100
Valid generations           : 99
Label accuracy              : 0.7172
Validation type accuracy    : 0.3939
Error type accuracy         : 0.4167
Correct-case accuracy       : 0.3590
Execution time              : 17.34 min


,theorem_id,name,hypotheses,expected,ground_truth_label,ground_truth_validation,ground_truth_reason,prediction_label,prediction_validation,prediction_reason,parse_error,raw_response
0,T21,Adjacent Sequences Theorem,"[(uₙ) is increasing and (vₙ) is decreasing, vₙ...","[(uₙ) is increasing and (vₙ) is decreasing, vₙ...",correct,exact_match,hypotheses match exactly.,correct,valid_implication,Both 'uₙ' increasing and 'vₙ' decreasing along...,None,"{""label"":""correct"",""validation_type"":""valid_im..."
1,T01,Rolle's Theorem,"[f is differentiable on ]a, b[]","[f is continuous on [a, b], f is differentiabl...",incorrect,multiple_missing,"missing: f is continuous on [a, b]; missing: f...",incorrect,invented_hypothesis,The proposed hypothesis 'f is differentiable o...,None,"{""label"":""incorrect"",""validation_type"":""invent..."
2,T24,Cauchy-Lipschitz Theorem,"[f is continuous in t, f is Lipschitz in y, an...","[f is continuous in t, f is Lipschitz in y, an...",correct,exact_match,hypotheses match exactly.,correct,exact_match,All required hypotheses are satisfied exactly.,None,"{""label"":""correct"",""validation_type"":""correct""..."
3,T22,Cauchy Criterion (sequences),[(uₙ) is a sequence in a complete space],[(uₙ) is a sequence in a complete space],correct,exact_match,hypotheses match exactly.,incorrect,missing_hypothesis,The Cauchy criterion requires that for every ε...,None,"{""label"":""incorrect"",""validation_type"":""missin..."
4,T03,Intermediate Value Theorem,"[f is of class C¹ on I, y is between f(a) and ...","[f is continuous on [a, b], y is between f(a) ...",correct,stronger_hypothesis,stronger_hypothesis: f is of class C¹ on I imp...,correct,valid_implication,"f is of class C¹ implies continuity, which is ...",None,"{""label"":""correct"",""validation_type"":""valid_im..."


In [4]:
valid = df[df["parse_error"].isna()].copy()

# 1. Le modèle est-il cohérent avec lui-même (label vs validation_type) ?
valid["self_consistent"] = valid.apply(
    lambda r: (r["prediction_label"] == "correct") == (r["prediction_validation"] in ["correct", "valid_implication"]),
    axis=1
)
print("Auto-cohérence du modèle:", valid["self_consistent"].mean())
print(valid[~valid["self_consistent"]][["prediction_label", "prediction_validation"]].head(10))

# 2. Le vocabulaire des réponses correspond-il exactement à ce qui est attendu ?
print("\nValeurs uniques produites par le modèle pour validation_type:")
print(valid["prediction_validation"].value_counts(dropna=False))

# 3. Matrice de confusion complète
print("\nCrosstab ground_truth vs prediction:")
print(pd.crosstab(valid["ground_truth_validation"].fillna("none"),
                   valid["prediction_validation"].fillna("none")))

Auto-cohérence du modèle: 0.7777777777777778
   prediction_label prediction_validation
2           correct           exact_match
5           correct           exact_match
8           correct           exact_match
11          correct           exact_match
12          correct           exact_match
13          correct           exact_match
17          correct           exact_match
18          correct           exact_match
22          correct           exact_match
23          correct           exact_match

Valeurs uniques produites par le modèle pour validation_type:
prediction_validation
valid_implication      31
invented_hypothesis    29
exact_match            22
missing_hypothesis     17
Name: count, dtype: int64

Crosstab ground_truth vs prediction:
prediction_validation    exact_match  invented_hypothesis  missing_hypothesis  \
ground_truth_validation                                                         
exact_match                       14                    0                   3 

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/mathurasanthalingam/benchmark/benchmark_data.json


In [2]:
import os
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/.virtual_documents/__notebook_source__.ipynb


### Si y'a des lignes qui n'ont pas pu etre parsée

In [ ]:
import pandas as pd
import time

# ============================================================
# PROMPTS
# ============================================================
SYSTEM_PROMPT = """You are a mathematics expert tasked with evaluating whether a given list of hypotheses for a theorem is correct — i.e., necessary and sufficient for the theorem to be applicable.

EVALUATION RULES:
1. A theorem has a set of hypotheses that are truly NECESSARY AND SUFFICIENT for it to be applicable (including, where relevant, problem data such as an initial condition, a domain of definition, a reference point, etc.).
2. Compare the proposed list of hypotheses to this actual required set.
3. A proposed hypothesis that is STRONGER than the actually required one is ACCEPTED, provided it logically implies it AND it REPLACES the required hypothesis (rather than being added alongside it).
   Example: if the theorem requires "f continuous" and the list REPLACES this with "f differentiable", this is CORRECT because differentiable ⇒ continuous, and no separate "f continuous" hypothesis is also present.
4. IMPORTANT — Redundant additions are NOT acceptable, even if logically valid: if the list already contains a hypothesis sufficient to satisfy a requirement, and an ADDITIONAL hypothesis is included that is merely implied by (redundant with) one already in the list, this extra hypothesis must be flagged as an invented/unnecessary hypothesis. Being mathematically true or logically implied does NOT make a hypothesis necessary in the list.
   Example: if the list contains "f is Lipschitz in y" (already sufficient) AND ALSO "f is continuous in y" (implied by the first), the second is redundant and must be flagged as "invented_hypothesis" — do NOT mark the list as correct just because the added hypothesis happens to be true.
5. Only take into account the hypotheses listed. Do not assume implicit hypotheses that are not mentioned as being "understood".
6. Do not discuss the mathematical truth of the theorem itself, only the completeness and correctness of the provided hypothesis list relative to the standard version of the theorem.

DECISION PROCEDURE (apply in this order):
a) List the actual necessary-and-sufficient hypotheses for the standard version of the theorem.
b) Check if every proposed hypothesis maps to a necessary one, either directly or as a valid REPLACEMENT (stronger version implying it, per rule 3).
c) Check if any proposed hypothesis is NOT needed to satisfy any requirement — i.e., it is redundant with another hypothesis already in the list, or entirely unrelated to the theorem. If so → "invented_hypothesis", regardless of whether it is mathematically true.
d) Check if any necessary hypothesis has no corresponding entry in the list at all → missing.
e) Only if every necessary hypothesis is covered exactly once (directly or via valid replacement) AND no extra/redundant hypothesis is present → label = "correct".

If the hypotheses are necessary and sufficient, the label is "correct" and validation_type must be null.

If the hypotheses are NOT correct, the label is "incorrect" and validation_type must be exactly one of the following categories (choose the single most fitting one):
- "missing_hypothesis": exactly one necessary hypothesis is missing.
- "multiple_missing": more than one necessary hypothesis is missing.
- "invented_hypothesis": a hypothesis that is not necessary for the theorem, it can be a hypothesis is present but stated imprecisely or incorrectly (e.g., wrong quantifier, wrong domain, wrong scope) so that it does not actually match what is required, or a  an extra hypothesis is added that is not necessary for the theorem. 


OUTPUT FORMAT (strict JSON, nothing else, no markdown, no explanation outside the JSON):
{
  "label": "correct" or "incorrect",
  "validation_type": null or one of ["missing_hypothesis", "multiple_missing", "invented_hypothesis"],
  "reason": "concise explanation of why the hypotheses are correct/sufficient, or which specific hypothesis is missing/wrong/extra and why, including relevant logical implications"
}
"""

USER_PROMPT_TEMPLATE = """Theorem: {theorem_name}

Proposed hypotheses:
{hypotheses_list}

Evaluate whether this list is necessary and sufficient.

Return ONLY the JSON.
"""


# ============================================================
# PARSER JSON
# ============================================================

def parse_llm_output(raw_text):

    if raw_text is None:
        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": "no_response",
        }

    cleaned = re.sub(
        r"```json|```",
        "",
        raw_text,
        flags=re.IGNORECASE,
    ).strip()

    try:

        parsed = json.loads(cleaned)

        return {
            "label": parsed.get("label"),
            "validation_type": parsed.get("validation_type"),
            "reason": parsed.get("reason"),
            "parse_error": None,
        }

    except Exception as e:

        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": str(e),
        }






def build_user_prompt(theorem_name, hypotheses):
    hyp = "\n".join(f"- {h}" for h in hypotheses)

    return USER_PROMPT_TEMPLATE.format(
        theorem_name=theorem_name,
        hypotheses_list=hyp,
    )



def call_llm(theorem_name, hypotheses):

    user_prompt = build_user_prompt(
        theorem_name,
        hypotheses,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return answer.strip()

# df doit être ton DataFrame en mémoire (celui produit par le run précédent)
# Si tu repars d'un CSV sauvegardé, décommente la ligne suivante :
df = pd.read_csv("/kaggle/working/benchmark_results.csv")

MAX_RETRIES_PER_ROW = 3  # nombre de tentatives de relance par ligne avant abandon

failed_mask = df["parse_error"].notna()
failed_indices = df[failed_mask].index

print(f"{len(failed_indices)} lignes à ré-exécuter.")

for idx in failed_indices:
    row = df.loc[idx]
    theorem_name = row["name"]

    # "hypotheses" peut être une liste (si df encore en mémoire) ou une string (si relu depuis CSV)
    hyps = row["hypotheses"]
    if isinstance(hyps, str):
        import ast
        hyps = ast.literal_eval(hyps)

    print(f"\nRé-exécution : {row['theorem_id']} - {theorem_name}")

    success = False
    for attempt in range(1, MAX_RETRIES_PER_ROW + 1):
        try:
            raw = call_llm(theorem_name, hyps)
            parsed = parse_llm_output(raw)

            if parsed["parse_error"] is None:
                # Succès : on met à jour la ligne dans df
                df.loc[idx, "prediction_label"] = parsed["label"]
                df.loc[idx, "prediction_error"] = parsed["validation_type"]
                df.loc[idx, "prediction_reason"] = parsed["reason"]
                df.loc[idx, "parse_error"] = None
                df.loc[idx, "raw_response"] = raw
                print(f"  -> Succès (tentative {attempt})")
                success = True
                break
            else:
                print(f"  -> Tentative {attempt} échouée (parse_error: {parsed['parse_error']})")

        except Exception as e:
            print(f"  -> Tentative {attempt} : erreur d'exécution ({e})")

        time.sleep(1)

    if not success:
        print(f"  -> ABANDON après {MAX_RETRIES_PER_ROW} tentatives pour {row['theorem_id']}")

# --- Sauvegarde du DataFrame mis à jour ---
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_results.csv"
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nDataFrame mis à jour sauvegardé dans {OUTPUT_CSV_PATH}")

# --- Vérification finale ---
remaining_failed = df["parse_error"].notna().sum()
print(f"Lignes encore en échec après ré-exécution : {remaining_failed}")